# Lab 01 · Preparing Data for a Language Model

**Time:** 60–90 minutes · **Prerequisites:** Python lists, slicing, and dictionaries · **Hardware:** CPU

Your task is to turn a small English corpus into batches for next-token prediction. By the end, you should be able to inspect a batch and explain where each target token came from.

Open this notebook in Jupyter, VS Code, or Google Colab. Run each demonstration, predict its output, then compare your prediction with the result. Exercise cells define starter functions; their check cells print a hint until you replace the `NotImplementedError` with your implementation. Solutions are at the end.

Parts 1–6 use the Python standard library and an embedded corpus. Part 7 is optional and uses the repository's tokenizer and storage format. The notebook does not train a model.

**Repository references:** [Step 1](https://github.com/FareedKhan-dev/train-llm-from-scratch#step-1-preparing-the-data) · [Pretraining data preparation](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/scripts/prepare_pretrain_data.py) · [Batch iterator](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/data_loader/data_loader.py)

## 1 · Define the prediction task

Suppose a corpus contains the sentence **The cat sat down.** If we temporarily treat each word as one token and omit punctuation, the text gives us three examples.

| Context | Next token |
|---|---|
| The | cat |
| The cat | sat |
| The cat sat | down |

The corpus supplies its own labels. At each position, the token immediately to the right is the target. This is the *self-supervised* objective used in causal language modeling.

The word example is for intuition. We will use a character tokenizer to inspect the mechanics and then compare it with the repository's byte-level BPE tokenizer in Part 7.

## 2 · Clean records and split by document

A JSONL corpus stores one JSON object per line. Each record here has a `text` field. The demonstration removes blank records and exact duplicates after trimming whitespace at the edges. It preserves case, punctuation, and whitespace inside the document.

Deduplicate and split at the document level before producing windows. Otherwise, two overlapping windows from the same document could end up in different splits. The fixed seed makes this demonstration repeatable. For a real corpus, near duplicates or documents from the same source may need to be grouped before splitting.

**Expected result:** 10 raw records → 8 unique documents → 6 training documents and 2 validation documents.

In [1]:
import json
import random
from pathlib import Path

RAW_JSONL = '''{"text": "The cat sat down by the window."}
{"text": "A small bird landed on the fence."}
{"text": "The rain stopped before noon."}
{"text": "Maya opened a book and began to read."}
{"text": "The soup was warm and smelled of garlic."}
{"text": "A green kite flew above the park."}
{"text": "We packed lunch for the walk."}
{"text": "The dog waited quietly by the door."}
{"text": "The cat sat down by the window."}
{"text": "   "}'''

raw_documents = [json.loads(line)["text"] for line in RAW_JSONL.splitlines()]
clean_documents = list(dict.fromkeys(text.strip() for text in raw_documents if text.strip()))
shuffled_documents = clean_documents.copy()
random.Random(42).shuffle(shuffled_documents)

train_documents = shuffled_documents[:6]
validation_documents = shuffled_documents[6:]

assert len(raw_documents) == 10
assert len(clean_documents) == 8
assert len(train_documents) == 6 and len(validation_documents) == 2
assert set(train_documents).isdisjoint(validation_documents)
print(f"Raw: {len(raw_documents)} | Unique: {len(clean_documents)}")
print(f"Train: {len(train_documents)} | Validation: {len(validation_documents)}")


raw_documents

Raw: 10 | Unique: 8
Train: 6 | Validation: 2


['The cat sat down by the window.',
 'A small bird landed on the fence.',
 'The rain stopped before noon.',
 'Maya opened a book and began to read.',
 'The soup was warm and smelled of garlic.',
 'A green kite flew above the park.',
 'We packed lunch for the walk.',
 'The dog waited quietly by the door.',
 'The cat sat down by the window.',
 '   ']

**Exercise 1.** Implement `clean_records` using a loop and a set. Strip each document, skip empty strings, and keep the first occurrence of each cleaned document. Preserve order. Run the check after editing the function.

In [2]:
def clean_records(documents):
    result = [doc.strip() for doc in documents if doc.strip()]
    return list(dict.fromkeys(result))

clean_records(["  Read. ", "", "Read.", "Read  more.", "read."])

['Read.', 'Read  more.', 'read.']

In [3]:
def check_clean_records(implementation):
    assert implementation(["  Read. ", "", "Read.", "Read  more.", "read."]) == [
        "Read.", "Read  more.", "read."
    ]
    assert implementation(raw_documents) == clean_documents

def check_exercise(name, check, implementation):
    try:
        check(implementation)
    except NotImplementedError:
        print(f"{name}: replace the starter code to run the check.")
    else:
        print(f"{name}: passed.")

check_exercise("Exercise 1", check_clean_records, clean_records)

Exercise 1: passed.


## 3 · Encode text into token IDs

A vocabulary maps symbols to integer IDs. The model later uses each ID to look up a learned embedding vector. The tokenizer in this section uses one character per token, including spaces and punctuation.

Build the vocabulary from the training split. Reserve ID 0 for unknown characters (`UNK`) and ID 1 for end-of-text (`EOT`). These reserved IDs belong to this demonstration. The repository's `r50k_base` uses EOT ID 50256.

Encode each document independently, append EOT, then concatenate the results into a flat stream. EOT gives the model a boundary to learn from. A training window may still contain tokens from either side of that boundary.

**Checkpoints:** A training document decodes back to its exact original text. Each split contains one EOT per document. Validation may contain characters outside the training vocabulary.

In [4]:
UNK_ID = 0
EOT_ID = 1

characters = sorted(set("".join(train_documents)))
character_to_id = {character: index for index, character in enumerate(characters, start=2)}
id_to_symbol = {token_id: character for character, token_id in character_to_id.items()}
id_to_symbol.update({UNK_ID: "<UNK>", EOT_ID: "<EOT>"})

def encode_characters(text):
    return [character_to_id.get(character, UNK_ID) for character in text]

def decode_characters(token_ids):
    return "".join(id_to_symbol[token_id] for token_id in token_ids)

def make_stream(documents):
    token_ids = []
    for document in documents:
        token_ids.extend(encode_characters(document))
        token_ids.append(EOT_ID)
    return token_ids

sample = train_documents[0]
sample_ids = encode_characters(sample)
assert decode_characters(sample_ids) == sample
print("Text:", repr(sample))
print("IDs: ", sample_ids)
print("Decoded:", repr(decode_characters(sample_ids)))

train_ids = make_stream(train_documents)
validation_ids = make_stream(validation_documents)
assert train_ids.count(EOT_ID) == len(train_documents)
assert validation_ids.count(EOT_ID) == len(validation_documents)

print("Stream excerpt:", repr(decode_characters(train_ids[:75])))
print("Train tokens:", len(train_ids))
print("Validation tokens:", len(validation_ids))
print("Unknown validation tokens:", validation_ids.count(UNK_ID))

Text: 'Maya opened a book and began to read.'
IDs:  [5, 8, 30, 8, 2, 21, 22, 12, 20, 12, 11, 2, 8, 2, 9, 21, 21, 17, 2, 8, 20, 11, 2, 9, 12, 14, 8, 20, 2, 26, 21, 2, 24, 12, 8, 11, 3]
Decoded: 'Maya opened a book and began to read.'
Stream excerpt: 'Maya opened a book and began to read.<EOT>The soup was warm and smelled of garl'
Train tokens: 209
Validation tokens: 66
Unknown validation tokens: 0


## 4 · Align input and target windows

For context length `T = 4`, take five tokens and make two overlapping slices.

| Source window | Input `x` | Target `y` |
|---|---|---|
| `[10, 20, 30, 40, 50]` | `[10, 20, 30, 40]` | `[20, 30, 40, 50]` |

At position 2 (zero-based), the model sees the prefix `[10, 20, 30]` and predicts `40`. Causal attention controls which input positions are visible. Here we prepare the inputs and labels. The general rules are `x[i] = token_ids[start + i]` and `y[i] = token_ids[start + i + 1]`.

**Exercise 2.** Implement `make_pair`. Raise `ValueError` for a negative start, a context length below 1, or a window shorter than `T + 1`. Before running the check, work out the expected slices for `[7, 3, 9, 2, 8, 4]` with `start=1` and `T=3`.

In [7]:
def make_pair(stream, start, context_length):
    if context_length < 1 or start < 0 or start + context_length >= len(stream):
        raise ValueError("Không đủ T+1 token hoặc tham số không hợp lệ")
    chunk = stream[start:start + context_length + 1]
    return chunk[:-1], chunk[1:]

In [8]:
def check_make_pair(implementation):
    ids = [7, 3, 9, 2, 8, 4]
    assert implementation(ids, 1, 3) == ([3, 9, 2], [9, 2, 8])
    assert implementation(ids, 2, 3) == ([9, 2, 8], [2, 8, 4])
    for start, context_length in [(-1, 3), (0, 0), (3, 3)]:
        try:
            implementation(ids, start, context_length)
        except ValueError:
            continue
        raise AssertionError(f"Expected ValueError for start={start}, T={context_length}")

check_exercise("Exercise 2", check_make_pair, make_pair)

Exercise 2: passed.


## 5 · Assemble a batch

A batch with size `B` and context length `T` has shape `(B, T)` for both inputs and targets. The repository's iterator chooses starts that are multiples of `T`, shuffles them, and reads `T + 1` tokens per row. This notebook uses Python lists to show the same indexing, while the repository returns PyTorch tensors.

For a stream of length `N`, starts are `0, T, 2T, ...` and must satisfy `start + T < N`. The number of complete windows with this stride is `(N - 1) // T` when `N >= 1`.

For example, with tokens `[0, 1, 2, 3, 4, 5, 6]` and `T=3`, the two complete windows start at 0 and 3. If the shuffled starts are `[3, 0]`, the batch is:

| Row | Start | Input | Target |
|---|---|---|---|
| 0 | 3 | `[3, 4, 5]` | `[4, 5, 6]` |
| 1 | 0 | `[0, 1, 2]` | `[1, 2, 3]` |

The rows are shuffled together with their labels. The order of tokens inside each row stays intact.

**Exercise 3.** Implement `make_batch` using the function from Exercise 2. Receive start indices explicitly and preserve their order. Return two lists of rows. After passing the checks, compare the number of available windows for `T=8` and `T=16`. Predict the change before running it.

In [22]:
def make_batch(token_ids, starts, context_length, pair_function):
    inputs = []
    targets = []
    for start in starts:
        input, target = pair_function(token_ids, start, context_length)
        
        inputs.append(input)
        targets.append(target)  
          
    return inputs, targets    

make_batch(list(range(12)), [4, 0], 4, make_pair)         

([[4, 5, 6, 7], [0, 1, 2, 3]], [[5, 6, 7, 8], [1, 2, 3, 4]])

In [23]:
def check_make_batch(implementation, pair_function):
    check_make_pair(pair_function)
    observed = implementation(list(range(12)), [4, 0], 4, pair_function)
    
    assert observed == ([[4, 5, 6, 7], [0, 1, 2, 3]], [[5, 6, 7, 8], [1, 2, 3, 4]])

check_exercise("Exercise 3", lambda fn: check_make_batch(fn, make_pair), make_batch)

Exercise 3: passed.


## 6 · Adapt the data for supervised fine-tuning

In the repository's chat setup, the SFT loss mask selects assistant content and the EOT that ends the assistant turn. The user prompt and role headers remain in the inputs so the model can use them as context.

If you create targets with `ids[1:]`, align the loss mask with those same target positions using `mask[1:]`. In the example below, the assistant header at input ID 14 predicts the first assistant token, ID 15. Because the target is assistant content, that prediction contributes to the loss.

| Data stream | Prepared example | Purpose |
|---|---|---|
| Pretraining | Flat stream of token IDs | Predict the next token in text |
| SFT | Chat token IDs with a loss mask | Predict assistant response tokens |
| Preference learning | `prompt`, `chosen`, `rejected` | Compare responses to the same prompt |
| Verifiable RL | `prompt`, `gold` | Score generated responses against a known answer |

**Exercise 4.** Predict which target IDs contribute to the loss. Then substitute `mask[:-1]` temporarily and identify which supervised prediction disappears.

In [27]:
chat_ids = [11, 12, 13, 14, 15, 16, EOT_ID]
loss_mask = [0, 0, 0, 0, 1, 1, 1]
inputs = chat_ids[:-1]
targets = chat_ids[1:]
target_mask = loss_mask[1:]

for input_id, target_id, is_supervised in zip(inputs, targets, target_mask):
    print(f"input={input_id:2}  target={target_id:2}  supervised={bool(is_supervised)}")

assert [token for token, flag in zip(targets, target_mask) if flag] == [15, 16, EOT_ID]

input=11  target=12  supervised=False
input=12  target=13  supervised=False
input=13  target=14  supervised=False
input=14  target=15  supervised=True
input=15  target=16  supervised=True
input=16  target= 1  supervised=True


## 7 · Optional bridge to the repository

In Jupyter or Colab, run `%pip install tiktoken h5py numpy` in a new cell. In a terminal, use `python -m pip install tiktoken h5py numpy` in the environment backing your notebook kernel. The first call to load `r50k_base` may need network access. Then set `RUN_REPO_FORMAT = True` and execute the cell.

The repository uses `encode_ordinary` for document text, appends EOT, and writes a one-dimensional HDF5 dataset named `tokens` with dtype `int32`. The lab writes separate training and validation files with the same dataset layout. It processes the embedded corpus instead of downloading The Pile.

**Experiment.** Compare token counts for the same English document under both tokenizers. Change `" The cat sat."` to `"The cat sat."` and inspect the BPE IDs. Spaces can be part of a token, so preserve whitespace that carries meaning. Different languages and writing systems may produce different tokens per character under the same tokenizer. Measure that ratio on representative samples before using English counts to budget context for another language.

In [28]:
RUN_REPO_FORMAT = False

if RUN_REPO_FORMAT:
    import h5py
    import numpy as np
    import tiktoken

    tokenizer = tiktoken.get_encoding("r50k_base")
    assert tokenizer.eot_token == 50256
    output_dir = Path("lab01_output")
    output_dir.mkdir(exist_ok=True)

    for split, documents in [("train", train_documents), ("val", validation_documents)]:
        token_ids = []
        for document in documents:
            document_ids = tokenizer.encode_ordinary(document)
            assert tokenizer.decode(document_ids) == document
            token_ids.extend(document_ids)
            token_ids.append(tokenizer.eot_token)

        array = np.asarray(token_ids, dtype=np.int32)
        output_path = output_dir / f"{split}.h5"
        with h5py.File(output_path, "w") as destination:
            destination.create_dataset("tokens", data=array)
        with h5py.File(output_path, "r") as source:
            dataset = source["tokens"]
            assert dataset.ndim == 1 and dataset.dtype == np.dtype("int32")
            assert np.array_equal(dataset[:], array)
            print(f"{output_path}: shape={dataset.shape}, dtype={dataset.dtype}")

    print("Character tokens in sample:", len(encode_characters(sample)))
    print("BPE tokens in sample:", len(tokenizer.encode_ordinary(sample)))
    for text in [" The cat sat.", "The cat sat."]:
        print(repr(text), tokenizer.encode_ordinary(text))
else:
    print("Optional section skipped. Set RUN_REPO_FORMAT to True after installing dependencies.")

Optional section skipped. Set RUN_REPO_FORMAT to True after installing dependencies.


## 8 · Reference solutions and end-to-end check

Work through the exercises before reading the code below. The cell runs the same checks against the reference implementations, then builds a real two-row batch and prints both rows in decoded form.

You're ready for Step 2 when you can point to one prediction in the printed batch, identify the exact prefix it uses, and trace its target to the next position in the source stream. Repeat that explanation for the first supervised assistant token in Part 6.

In [29]:
def clean_records_solution(documents):
    seen = set()
    result = []
    for document in documents:
        text = document.strip()
        if text and text not in seen:
            seen.add(text)
            result.append(text)
    return result

def make_pair_solution(token_ids, start, context_length):
    if context_length < 1 or start < 0 or start + context_length >= len(token_ids):
        raise ValueError("A valid window must contain T + 1 tokens.")
    window = token_ids[start:start + context_length + 1]
    return window[:-1], window[1:]

def make_batch_solution(token_ids, starts, context_length, pair_function):
    inputs, targets = [], []
    for start in starts:
        x, y = pair_function(token_ids, start, context_length)
        inputs.append(x)
        targets.append(y)
    return inputs, targets

check_clean_records(clean_records_solution)
check_make_pair(make_pair_solution)
check_make_batch(make_batch_solution, make_pair_solution)

batch_size, context_length = 2, 8
starts = list(range(0, len(train_ids) - context_length, context_length))
random.Random(42).shuffle(starts)
batch_starts = starts[:batch_size]
x, y = make_batch_solution(train_ids, batch_starts, context_length, make_pair_solution)

assert len(x) == len(y) == batch_size
for start, input_row, target_row in zip(batch_starts, x, y):
    assert len(input_row) == len(target_row) == context_length
    assert input_row[1:] == target_row[:-1]
    assert target_row[-1] == train_ids[start + context_length]
    print(f"Start {start:3} | x={decode_characters(input_row)!r} | y={decode_characters(target_row)!r}")

for length in (8, 16):
    count = (len(train_ids) - 1) // length
    print(f"T={length}: {count} complete windows")

Start 128 | x='tly by t' | y='ly by th'
Start  96 | x='or the w' | y='r the wa'
T=8: 26 complete windows
T=16: 13 complete windows


**Exercise 4 solution.** The correct supervised targets are 15, 16, and EOT. With `mask[:-1]`, only 16 and EOT contribute. The first assistant token is dropped because its mask entry was read from the previous input position.